[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/Multivariate_Occupancy_RNN_AdvancedTopics.ipynb)

# Multivariate Example (Advanced Topics)
**Dr. Dave Wanik - University of Connecticut**

Let's see if we can predict whether a room is occupied as a function of its environmental data - let's see if Conv1D and MaxPooling1D can make an even better prediction (ConvLSTM).

**Common errors:** forgetting an activation function, not specifying the input shape (or mixing up the order!), not returning sequences when two layers (or accidentally return_sequences=True when only one layer!)


In [1]:
# standard modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# RNN-specific modules
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report,accuracy_score
from tensorflow.keras import layers, Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, GRU, LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

## Read in data
Check for missing values, make some plots.

In [2]:
# Dataset initially sourced from LuisM78’s GitHub repository:
# url = 'https://raw.githubusercontent.com/LuisM78/Occupancy-detection-data/master/datatest.txt'
# Link to the data file on Github
url = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/refs/heads/main/OPIM5509_Module4_Files/data/datatest.txt"

# read the data
df = pd.read_csv(url)
print(df.info())
df.head(n=15) # nice complete data! this will allow us to check our work later

<class 'pandas.DataFrame'>
RangeIndex: 2665 entries, 140 to 2804
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           2665 non-null   str    
 1   Temperature    2665 non-null   float64
 2   Humidity       2665 non-null   float64
 3   Light          2665 non-null   float64
 4   CO2            2665 non-null   float64
 5   HumidityRatio  2665 non-null   float64
 6   Occupancy      2665 non-null   int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 195.3 KB
None


,date,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
140,2015-02-02 14:19:00,23.7000,26.272,585.200000,749.200000,0.004764,1
141,2015-02-02 14:19:59,23.7180,26.290,578.400000,760.400000,0.004773,1
142,2015-02-02 14:21:00,23.7300,26.230,572.666667,769.666667,0.004765,1
143,2015-02-02 14:22:00,23.7225,26.125,493.750000,774.750000,0.004744,1
144,2015-02-02 14:23:00,23.7540,26.200,488.600000,779.000000,0.004767,1
145,2015-02-02 14:23:59,23.7600,26.260,568.666667,790.000000,0.004779,1
146,2015-02-02 14:25:00,23.7300,26.290,536.333333,798.000000,0.004776,1
147,2015-02-02 14:25:59,23.7540,26.290,509.000000,797.000000,0.004783,1
148,2015-02-02 14:26:59,23.7540,26.350,476.000000,803.200000,0.004794,1
149,2015-02-02 14:28:00,23.7360,26.390,510.000000,809.000000,0.004796,1


In [3]:
# count of occupancy
df['Occupancy'].value_counts() # not perfectly balanced, but that's OK

Occupancy
0    1693
1     972
Name: count, dtype: int64

In [4]:
# visualize the data
df['Occupancy'].plot()
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_36200\633319714.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# visualize the data
df['CO2'].plot()
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_36200\165701153.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# drop the date column
df.drop(['date'], inplace=True, axis=1)
print(df.shape)
df.head()

(2665, 6)


,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
140,23.7000,26.272,585.200000,749.200000,0.004764,1
141,23.7180,26.290,578.400000,760.400000,0.004773,1
142,23.7300,26.230,572.666667,769.666667,0.004765,1
143,23.7225,26.125,493.750000,774.750000,0.004744,1
144,23.7540,26.200,488.600000,779.000000,0.004767,1


In [7]:
# prep data for modeling (multivariate)
# link: https://machinelearningmastery.com/how-to-develop-lstm-models-for-time-series-forecasting/

from numpy import array

# split a multivariate sequence into samples
def split_sequences(sequences, n_steps):
	X, y = list(), list()
	for i in np.arange(len(sequences)): # be careful of this line!
		# find the end of this pattern
		end_ix = i + n_steps
		# check if we are beyond the dataset
		if end_ix > len(sequences):
			break
		# gather input and output parts of the pattern
		seq_x, seq_y = sequences[i:end_ix, :-1], sequences[end_ix-1, -1]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

In [8]:
# we could split our data first, normalize it, then create sequences

In [9]:
# all we need to do is decide on is n_steps (what our lookback period is)
# since we have a bunch of data, why not n_steps=10? then try 30 later on.
n_steps = 50
raw_seq = np.array(df) #make sure your data is stored as a numpy array!
# let's ignore the date column and just use the temperature data
X, y = split_sequences(raw_seq, n_steps=50)

In [10]:
# take a peak at what it did
print(X.shape)
print(y.shape)

# scroll up and make sure you understand this!
# y is a function of X (the previous n_steps observations!)

(2616, 50, 5)
(2616,)


In [11]:
# split the data into train and test partitions
# we will use 50% of the data for train, and 50% for validation
train_pct_index = int(0.5 * len(X))
X_train, X_test = X[:train_pct_index], X[train_pct_index:]
y_train, y_test = y[:train_pct_index], y[train_pct_index:]

# pretty slick way of splitting your data using slicing!
# notice how we didn't do any shuffling (we don't want temporal leakage! keeps time series intact)

In [12]:
# check the shape to be sure
print(X.shape, X_train.shape, X_test.shape)
print(y.shape, y_train.shape, y_test.shape)

# verify that this all adds up!
# 2635 samples with 30 lookback and 6 columns

(2616, 50, 5) (1308, 50, 5) (1308, 50, 5)
(2616,) (1308,) (1308,)


# RNN one layer model (with Conv1D and pooling!)
With Conv1D and MaxPooling1D... don't forget that input shape needs to go in the first layer!

In [13]:
n_steps = X_train.shape[1] # lookback
n_features = X_train.shape[2] # columns

print(n_steps, n_features)

50 5


In [14]:
# now let's build a model

# define
n_steps = X_train.shape[1]
n_features = X_train.shape[2]


# define model
model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(SimpleRNN(30, activation='relu', recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=5,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 48, 32)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 30)             │         1,890 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433 (9.50 KB)

 Trainable params: 2,433 (9.50 KB)

...

198/210 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - acc: 0.9747 - loss: 0.0944

205/210 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - acc: 0.9756 - loss: 0.0921

210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - acc: 0.9761 - loss: 0.0904 - val_acc: 0.9771 - val_loss: 0.1107


Epoch 11: early stopping


Restoring model weights from the end of the best epoch: 1.


In [15]:
# show scatterplots of actual vs. predicted for train and test
# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
pred # run all if you get an error...

 1/41 ━━━━━━━━━━━━━━━━━━━━ 14s 368ms/step

19/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step   

38/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step


[[1.]
 [1.]
 [1.]
 ...
 [1.]
 [1.]
 [1.]]


array([[1.],
       [1.],
       [1.],
       ...,
       [1.],
       [1.],
       [1.]], shape=(1308, 1), dtype=float32)

In [16]:
# confusion matrix
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

[[824  25]
 [  1 458]]
              precision    recall  f1-score   support

         0.0       1.00      0.97      0.98       849
         1.0       0.95      1.00      0.97       459

    accuracy                           0.98      1308
   macro avg       0.97      0.98      0.98      1308
weighted avg       0.98      0.98      0.98      1308



In [17]:
# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_36200\774347038.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

 1/41 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step

20/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 

40/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


[[1.]
 [1.]
 [1.]
 ...
 [1.]
 [1.]
 [1.]]
[[1.]
 [1.]
 [1.]
 ...
 [1.]
 [1.]
 [1.]]
[[824  25]
 [  1 458]]
              precision    recall  f1-score   support

         0.0       1.00      0.97      0.98       849
         1.0       0.95      1.00      0.97       459

    accuracy                           0.98      1308
   macro avg       0.97      0.98      0.98      1308
weighted avg       0.98      0.98      0.98      1308



C:\Users\dww05002\AppData\Local\Temp\ipykernel_36200\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# RNN two layer model
Don't forget to set return_sequences=True!

In [19]:
# now let's build a model

# define
n_steps = X_train.shape[1]
n_features = X_train.shape[2]


# define model
model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(Bidirectional(SimpleRNN(30, return_sequences=True, recurrent_dropout=0.2))) # don't forget to return_sequences!
model.add(Bidirectional(SimpleRNN(30, recurrent_dropout=0.2)))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=5,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 48, 32)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 60)         │         3,780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 60)             │         5,460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            61 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,813 (38.33 KB)

 Trainable params: 9,813 (38.33 KB)

...

207/210 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - acc: 0.9739 - loss: 0.0955

210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - acc: 0.9742 - loss: 0.0945

210/210 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - acc: 0.9742 - loss: 0.0945 - val_acc: 0.9771 - val_loss: 0.1344


Epoch 11: early stopping


Restoring model weights from the end of the best epoch: 1.


In [20]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

 1/41 ━━━━━━━━━━━━━━━━━━━━ 38s 957ms/step

10/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step   

19/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

28/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

37/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step


[[0.89254713]
 [0.8925893 ]
 [0.89255244]
 ...
 [0.91341794]
 [0.9134065 ]
 [0.9134327 ]]
[[1.]
 [1.]
 [1.]
 ...
 [1.]
 [1.]
 [1.]]
[[804  45]
 [  7 452]]
              precision    recall  f1-score   support

         0.0       0.99      0.95      0.97       849
         1.0       0.91      0.98      0.95       459

    accuracy                           0.96      1308
   macro avg       0.95      0.97      0.96      1308
weighted avg       0.96      0.96      0.96      1308



C:\Users\dww05002\AppData\Local\Temp\ipykernel_36200\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# LSTM one layer model
Literally, just grab the code above and change SimpleRNN to LSTM and boom! you have a more sophisticated model.

In [21]:
# define shape
n_steps = X_train.shape[1]
n_features = X_train.shape[2]


# define model
model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D())
model.add(LSTM(30, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=5,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 48, 32)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 30)             │         7,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,103 (31.65 KB)

 Trainable params: 8,103 (31.65 KB)

...

199/210 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - acc: 0.8010 - loss: 0.5054

205/210 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - acc: 0.8029 - loss: 0.5032

210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8040 - loss: 0.5020 - val_acc: 0.0229 - val_loss: 1.3836


Epoch 11: early stopping


Restoring model weights from the end of the best epoch: 1.


In [22]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

 1/41 ━━━━━━━━━━━━━━━━━━━━ 13s 349ms/step

13/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step   

26/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

39/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


[[0.5886785 ]
 [0.6291187 ]
 [0.6119151 ]
 ...
 [0.7392143 ]
 [0.49463767]
 [0.46719462]]
[[1.]
 [1.]
 [1.]
 ...
 [1.]
 [0.]
 [0.]]
[[799  50]
 [228 231]]
              precision    recall  f1-score   support

         0.0       0.78      0.94      0.85       849
         1.0       0.82      0.50      0.62       459

    accuracy                           0.79      1308
   macro avg       0.80      0.72      0.74      1308
weighted avg       0.79      0.79      0.77      1308



C:\Users\dww05002\AppData\Local\Temp\ipykernel_36200\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# LSTM two layer model

In [23]:
# now let's build a model

# since this is a univariate problem, n_features will be 1 (we also defined this before)

# define model
model = Sequential()
model.add(Conv1D(filters=128, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(Bidirectional(LSTM(30,
                            return_sequences=True, # remember, if stacking layers, you need to return sequences!
                            activation='relu',
                            recurrent_dropout=0.2)))
model.add(GRU(20, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=5,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 48, 128)        │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 24, 60)         │        38,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20)             │         4,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,149 (176.36 KB)

 Trainable params: 45,149 (176.36 KB)

...

207/210 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - acc: 0.7517 - loss: 2.9913

209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - acc: 0.7531 - loss: 2.9658

210/210 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - acc: 0.7533 - loss: 2.9633 - val_acc: 0.0229 - val_loss: 1.2162


Epoch 12: early stopping


Restoring model weights from the end of the best epoch: 2.


In [24]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

 1/41 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step

 7/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

13/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

20/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

26/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

...

38/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step

41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step


[[9.9961185e-01]
 [9.9976557e-01]
 [9.7674388e-04]
 ...
 [9.9900854e-01]
 [9.9918532e-01]
 [9.9936867e-01]]
[[1.]
 [1.]
 [0.]
 ...
 [1.]
 [1.]
 [1.]]
[[822  27]
 [355 104]]
              precision    recall  f1-score   support

         0.0       0.70      0.97      0.81       849
         1.0       0.79      0.23      0.35       459

    accuracy                           0.71      1308
   macro avg       0.75      0.60      0.58      1308
weighted avg       0.73      0.71      0.65      1308



C:\Users\dww05002\AppData\Local\Temp\ipykernel_36200\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
